# Real-Time IoT Analytics Pipeline

In [ ]:
!docker-compose up -d

In [7]:
!docker ps

CONTAINER ID   IMAGE                             COMMAND                  CREATED          STATUS                    PORTS                                                                            NAMES
7ee24aaba863   cassandra:4.1                     "docker-entrypoint.s…"   30 seconds ago   Up 27 seconds             7000-7001/tcp, 7199/tcp, 9042/tcp, 9160/tcp                                      cassandra-3
5fce245e332f   cassandra:4.1                     "docker-entrypoint.s…"   30 seconds ago   Up 27 seconds             7000-7001/tcp, 7199/tcp, 9042/tcp, 9160/tcp                                      cassandra-2
ae81ac2f77fd   confluentinc/cp-kafka:7.5.0       "/etc/confluent/dock…"   37 seconds ago   Up 36 seconds             0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp                                      kafka
f7a4815a4327   cassandra:4.1                     "docker-entrypoint.s…"   38 seconds ago   Up 37 seconds (healthy)   7000-7001/tcp, 7199/tcp, 9160/tcp, 0.0.0.0:9042->9042/t

## Part 1: The Pipeline Overview (2-3 minutes)

### System Architecture

```
IoT Sensors (100 devices)
    ↓ 100 events/sec
┌──────────────────────────────┐
│  Apache Kafka (Message Broker)│  - Decoupling
│  sensor-events topic          │  - Buffering
└──────────────────────────────┘
    ↓
┌──────────────────────────────┐
│  Apache Spark Streaming       │  - Real-time aggregation
│  (micro-batch processing)    │  - Hourly statistics
└──────────────────────────────┘
    ↓
┌──────────────────────────────┐
│  Cassandra 3-Node Cluster    │  - Write-optimized storage
│  RF=3 (Replication Factor)   │  - Distributed architecture
└──────────────────────────────┘
```

### The Three Components

| Component | Role | Benefit |
|-----------|------|----------|
| **Kafka** | Ingestion & Buffering | Decouples producers from consumers, prevents data loss |
| **Spark** | Stream Processing | Real-time aggregation, stateful transformations |
| **Cassandra** | Distributed Storage | Handles velocity, volume; tunable consistency; fault-tolerant |

#### Check cluster health

In [18]:
!docker exec -it cassandra-1 nodetool status

Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load      Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.6  2.43 MiB  16      100.0%            6d110fa3-3de5-404d-b6a6-6c31e392628f  rack1
UN  172.18.0.3  2.68 MiB  16      100.0%            c9cee0af-7ef6-4b8f-b5a0-1e7f149c35d4  rack1
UN  172.18.0.5  2.42 MiB  16      100.0%            e10dedf1-ad6d-4456-a378-3fd7f5fe6042  rack1



#### Start Producer and Consumer 

In [ ]:
# Start producer
python3 producer.py

# Start Spark consumer
spark-submit \
  --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,com.datastax.spark:spark-cassandra-connector_2.12:3.5.0 \
  --conf spark.cassandra.connection.host=localhost \
  --conf spark.cassandra.connection.port=9042 \
  spark_consumer.py

## Part 2: The Big Data Challenges (1 minute)

### The 4 Vs of Big Data

| Challenge | Definition | Cassandra Solution |
|-----------|------------|-----------|
| **Volume** | Massive datasets (TBs, PBs) | **Scale-out**: Add nodes horizontally |
| **Velocity** | Fast data production (streaming) | **Write-optimized**: Append-only log, no updates. **Tunable consistency** |
| **Variety** | Heterogeneous formats & schemas | **Schemaless**: Flexible, no rigid schema |
| **Veracity** | Data quality issues | Cassandra **does not solve** this, handled in application logic |

### Traditional Databases Fail

- **Relational DBs**: Not designed for distributed clusters, single node.
- **Scale-up limit**: Single machine bottleneck
- **ACID burden**: Strong consistency but too strict for distributed systems
- **Network latency**: Coordination overhead destroys throughput

**NoSQL approach**: Trade consistency for availability & partition tolerance

## Part 3: Cassandra Deep Dive
### Cassandra Architecture: Masterless Design

![./images/master-vs-p2p.png](./images/master-vs-p2p.png)

```
Problems:                   Benefits:
✗ Master failure            ✓ No single point of failure
✗ Write bottleneck          ✓ Linear scalability
✗ Can't scale writes        ✓ Any node = coordinator
```

### CAP Theorem: Cassandra's Trade-offs

```
Cassandra chooses: A + P (Availability + Partition Tolerance)

  C = Consistency (all replicas always agree)
  A = Availability (respond to requests)
  P = Partition Tolerance (survive network failures)

┌─────────────────────────────────────┐
│  You can have at most 2 of 3       │
│                                     │
│  SQL:  C + A (single node)         │
│  Mongo: C + P (strong cons., some latency)          │
│  Cassandra: A + P (eventual)       │
└─────────────────────────────────────┘
```

![./images/master-vs-p2p.png](./images/CAP.png)

**Why not choose C?**
- Network partitions **always** happen (reality)
- Strict consistency requires coordination → latency & unavailability
- For high-volume, low-latency systems: availability wins

### Cassandra's Core Characteristics

#### 1️⃣ **NoSQL Column-Family Model**

```
sensor_events table:

Partition Key     | Clustering Key   | Data
device_id         | timestamp        | temp, humidity, location
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
uuid-001          | 1704067200       | 22.5°C, 45%, kitchen
uuid-001          | 1704067201       | 22.6°C, 44%, kitchen
uuid-001          | 1704067202       | 22.7°C, 43%, kitchen
uuid-002          | 1704067200       | 18.2°C, 52%, bedroom
```

**Key Features**:
- **Partition Key** (uuid-001): Determines node placement via hash ring
- **Clustering Key** (timestamp): Sorts rows within partition (DESC = newest first)
- **Columns**: Dynamic, sparse; only store what exists

#### 2️⃣ **Distributed Hash Ring**

![./images/master-vs-p2p.png](./images/cassandra-vnodes.png)

```
Token assignment (vnodes = 16 tokens/node):
• UUID hash → token value (0-2^127)
• Token range determines replica placement
• RF=3 means token is replicated on 3 consecutive nodes
• Gossip protocol: nodes exchange heartbeats every second
  → Auto-detects failures, no ZooKeeper needed
```

#### 3️⃣ **Tunable Consistency (The Secret Sauce)**

```
Write operation with CL=ONE:
Client → [Coordinator Node]
                ↓
            Replicas: [Node1, Node2, Node3]
                      ✓    (writes only to 1)
            Returns: "Written" immediately
                      (other replicas sync later)

Read operation with CL=QUORUM (2 out of 3):
Client → [Coordinator Node]
                ↓
            Reads from: [Node1, Node2, Node3]
                        ✓      ✓      ✗
            Quorum met (2/3) → Returns: newer value
                      (repair Node3 in background)
```

| Consistency Level | Write | Read | Use Case |
|-------------------|-------|------|----------|
| **ONE** | 1 replica | 1 replica | Max speed, accept stale reads |
| **QUORUM** | ⌈n/2⌉ | ⌈n/2⌉ | Balanced: strong consistency at lower latency |
| **ALL** | all | all | Max consistency, slower, less available |

**Our Project Strategy**: Write at ONE (fast ingestion) + Read at QUORUM (verify correctness)

#### 4️⃣ **Eventual Consistency & Repair**

```
Scenario: Write with CL=ONE during network partition

Time 0:   ┌─────────┬─────────┬─────────┐
          │ Node A  │ Node B  │ Node C  │
          │  v=10   │  v=10   │  v=10   │
          └─────────┴─────────┴─────────┘

Time 1:   Client writes: sensor_temp = 22.5
          CL=ONE → writes only to Node A

          ┌─────────┬─────────┬─────────┐
          │ Node A  │ Node B  │ Node C  │
          │  v=22.5 │  v=10   │  v=10   │  ← Inconsistent!
          └─────────┴─────────┴─────────┘

Time 2:   Read with CL=QUORUM
          [Node A v=22.5, Node B v=10] → newer wins
          Background repair: Node C.v = 22.5

Time 3:   ┌─────────┬─────────┬─────────┐
          │ Node A  │ Node B  │ Node C  │
          │  v=22.5 │  v=22.5 │  v=22.5 │  ← Consistent!
          └─────────┴─────────┴─────────┘
```

**This is "Eventual Consistency"**: System reaches consistency state eventually

#### 5️⃣ **Scale-Out Architecture**

```
Traditional Relational DB:     Cassandra Cluster:

Single Server (Scale-Up)       Multiple Nodes (Scale-Out)
┌──────────────────────┐       ┌──────┐  ┌──────┐  ┌──────┐
│ Relational DB        │       │Node1 │  │Node2 │  │Node3 │
│ ├─CPU (max: 256)     │       └──────┘  └──────┘  └──────┘
│ ├─RAM (max: 2TB)     │       ├─CPU×N   ├─CPU×N   ├─CPU×N
│ ├─Disk (max: 1PB)    │       ├─RAM×N   ├─RAM×N   ├─RAM×N
│ └─Network: SPOF      │       └─Disk×N  └─Disk×N  └─Disk×N
└──────────────────────┘       
Hit ceiling ~1M qps            Linear scaling: add node → +1M qps
```

**Shared-Nothing**: Each node owns data independently
- No shared disk bottleneck
- No shared memory coherency
- Nodes communicate via network only

### Tackling Big Data Challenges: Cassandra Solutions

| Big Data Challenge | Cassandra Feature | How It Works |
|------------------|-------------------|---------------|
| **Volume** | Horizontal scaling | Add nodes; data redistributes via token ring |
| **Velocity** | Write-optimized storage | Append-only SSTables; no read-before-write |
| **Variety** | Schemaless/column-family | Add columns dynamically; sparse storage |
| **Veracity** | Tunable consistency + repair | Read at QUORUM; background anti-entropy repair |

### Compaction Strategies (Performance Tuning)

**Why?** Cassandra writes always append (fast), but reads slow as files accumulate. Compaction merges files.

| Strategy | Write Speed | Read Speed | Best For | Trade-off |
|----------|------------|-----------|----------|----------|
| **SizeTiered** | ⚡⚡ Ultra-fast | Slow | Logs, time-series (write-heavy) | High read latency |
| **Leveled** | Slower | ⚡⚡ Fast | Analytics queries | Compaction overhead |
| **TimeWindow** | ⚡ Fast | ⚡ Balanced | Events with TTL expiry | Date-based organization |

**Our Project**: Uses all three strategies to demonstrate trade-offs!

## Part 4: Live Demonstration (3-5 minutes)

### What We'll Show

1. **Cluster Status**: Verify 3-node ring is healthy
2. **Data Ingestion**: Show Kafka → Spark → Cassandra flow
3. **Consistency Levels**: Compare ONE vs QUORUM reads
4. **Load Balancing**: Add 4th node and watch data rebalance

### Startup Sequence

In [ ]:
# Environment activation
source /home/alessio/Documents/cassandra-iot-pipeline/.venv/bin/activate

# Navigate to project directory
cd Documents/cassandra-iot-pipeline

# Start Docker infrastructure (Cassandra, Kafka, Zookeeper)
docker-compose up -d

# Wait for Cassandra cluster formation
docker exec cassandra-1 nodetool status

# Start producer
python3 producer.py

# Start Spark consumer
spark-submit \
  --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,com.datastax.spark:spark-cassandra-connector_2.12:3.5.0 \
  --conf spark.cassandra.connection.host=localhost \
  --conf spark.cassandra.connection.port=9042 \
  spark_consumer.py

In [26]:
!docker ps -a

CONTAINER ID   IMAGE                             COMMAND                  CREATED          STATUS                 PORTS                                                                            NAMES
e92a91803701   cassandra:4.1                     "docker-entrypoint.s…"   11 seconds ago   Up 10 seconds          7000-7001/tcp, 7199/tcp, 9042/tcp, 9160/tcp                                      cassandra-4
5770c5d2f88b   cassandra:4.1                     "docker-entrypoint.s…"   2 hours ago      Up 2 hours             7000-7001/tcp, 7199/tcp, 9042/tcp, 9160/tcp                                      cassandra-3
4ea62f267fac   cassandra:4.1                     "docker-entrypoint.s…"   2 hours ago      Up 2 hours             7000-7001/tcp, 7199/tcp, 9042/tcp, 9160/tcp                                      cassandra-2
b4e50a0b3701   confluentinc/cp-kafka:7.5.0       "/etc/confluent/dock…"   2 hours ago      Up 2 hours             0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp                     

In [33]:
# Wait for Cassandra cluster formation
!docker exec cassandra-1 nodetool status

Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load      Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.6  2.62 MiB  16      100.0%            6d110fa3-3de5-404d-b6a6-6c31e392628f  rack1
UN  172.18.0.3  3.01 MiB  16      100.0%            c9cee0af-7ef6-4b8f-b5a0-1e7f149c35d4  rack1
UN  172.18.0.5  3.21 MiB  16      100.0%            e10dedf1-ad6d-4456-a378-3fd7f5fe6042  rack1



#### Start 4-th node

In [24]:
%%bash

echo ""
echo "Starting 4th node..."
docker run --name cassandra-4 \
  --network iot-cassandra-pipeline_iot-network \
  -m 1g \
  -e CASSANDRA_SEEDS=cassandra-1 \
  -e CASSANDRA_CLUSTER_NAME="IoT-Cluster" \
  -e CASSANDRA_DC=dc1 \
  -e CASSANDRA_RACK=rack1 \
  -e CASSANDRA_ENDPOINT_SNITCH=GossipingPropertyFileSnitch \
  -e MAX_HEAP_SIZE="512M" \
  -e HEAP_NEWSIZE="100M" \
  -d cassandra:4.1


Starting 4th node...
e92a91803701bb59874d03072d1ab657ef639a12dc6fee2ed3e717cf2a918a19


### Demo 1: Cluster Status & Token Ring

In [17]:
%%bash

#!/bin/bash
# Show cluster health: nodetool status

echo "=== CASSANDRA CLUSTER STATUS ==="
echo ""
echo "Command: nodetool status"
echo "Shows: Which nodes are Up/Normal and their token ownership"
echo ""

docker exec cassandra-1 nodetool status

echo ""
echo "Interpretation:"
echo "  UN = Up/Normal (healthy)"
echo "  16 tokens = Virtual nodes (vnodes) for load distribution"
echo "  100.0% ownership = With RF=3, each node owns 100% of the ring"
echo "                     (because data is replicated 3x)"
echo ""
echo "Key Insight: Token range determines which node stores each partition key"

=== CASSANDRA CLUSTER STATUS ===

Command: nodetool status
Shows: Which nodes are Up/Normal and their token ownership

Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load      Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.6  2.43 MiB  16      100.0%            6d110fa3-3de5-404d-b6a6-6c31e392628f  rack1
UN  172.18.0.3  2.68 MiB  16      100.0%            c9cee0af-7ef6-4b8f-b5a0-1e7f149c35d4  rack1
UN  172.18.0.5  2.42 MiB  16      100.0%            e10dedf1-ad6d-4456-a378-3fd7f5fe6042  rack1


Interpretation:
  UN = Up/Normal (healthy)
  16 tokens = Virtual nodes (vnodes) for load distribution
  100.0% ownership = With RF=3, each node owns 100% of the ring
                     (because data is replicated 3x)

Key Insight: Token range determines which node stores each partition key


### Demo 2: Tunable Consistency in Action

Key Takeaway
* CL=ONE + CL=QUORUM = Strong eventual consistency
* Write at ONE: 100+ μs (microseconds) -> Velocity on massive dataset or data streaming
* Read at QUORUM: 1-2 ms (milliseconds) -> Weak but Eventual Consistency

In [60]:
import time
from cassandra.cluster import Cluster
from cassandra import ConsistencyLevel

cluster = Cluster(['127.0.0.1'])
session = cluster.connect('iot_analytics')

def run_benchmark(cl_name, cl_value, iterations=100):
    latencies = []
    # Set the consistency level for the session
    session.default_consistency_level = cl_value
    
    for _ in range(iterations):
        start = time.perf_counter()
        
        # We use a query that forces the coordinator to check replicas
        # Note: Large LIMITs or broad scans are best for seeing coordination overhead
        result = session.execute("SELECT * FROM sensor_events LIMIT 1000")
        list(result) # Force full data fetching
        
        end = time.perf_counter()
        latencies.append((end - start) * 1000)
    
    avg_latency = sum(latencies) / len(latencies)
    print(f"Consistency {cl_name:8}: Avg {avg_latency:.2f} ms")

# Running the tests
print("Starting Latency Tests...")
run_benchmark("ONE", ConsistencyLevel.ONE)
run_benchmark("QUORUM", ConsistencyLevel.QUORUM)
run_benchmark("ALL", ConsistencyLevel.ALL)

Starting Latency Tests...
Consistency ONE     : Avg 25.38 ms
Consistency QUORUM  : Avg 33.12 ms
Consistency ALL     : Avg 38.41 ms


### Demo 3: Add 4th Node & Watch Load Balancing

In [61]:
%%bash

#!/bin/bash
# Demonstrate horizontal scaling

echo ""
echo "After: 4 nodes, data redistributes automatically!"
docker exec cassandra-1 nodetool status

echo ""
echo "What happened:"
echo "  1. New node joins the cluster (gossip protocol)"
echo "  2. Token ring recalculates partition ownership"
echo "  3. Data streams from old nodes to new node"
echo "  4. No downtime, no manual rebalancing!"
echo ""
echo "This is Cassandra's 'Linear Scalability': add node → system rebalances"


After: 4 nodes, data redistributes automatically!
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load      Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.6  2.91 MiB  16      100.0%            6d110fa3-3de5-404d-b6a6-6c31e392628f  rack1
UN  172.18.0.3  3.01 MiB  16      100.0%            c9cee0af-7ef6-4b8f-b5a0-1e7f149c35d4  rack1
UN  172.18.0.5  3.21 MiB  16      100.0%            e10dedf1-ad6d-4456-a378-3fd7f5fe6042  rack1


What happened:
  1. New node joins the cluster (gossip protocol)
  2. Token ring recalculates partition ownership
  3. Data streams from old nodes to new node
  4. No downtime, no manual rebalancing!

This is Cassandra's 'Linear Scalability': add node → system rebalances


### Demo 4: View Sample Data

In [62]:
%%bash

#!/bin/bash
# Show actual data

echo "=== RAW DATA IN CASSANDRA ==="
echo ""
echo "Raw sensor events (recent first):"
cat <<'EOF' | docker exec -i cassandra-1 cqlsh
USE iot_analytics;
SELECT device_id, timestamp, temperature, humidity, location 
FROM sensor_events LIMIT 10;
EOF

echo "=== RAW DATA IN CASSANDRA ==="
echo ""
echo "Raw sensor events (recent first):"
cat <<'EOF' | docker exec -i cassandra-1 cqlsh
USE iot_analytics;
SELECT * FROM sensor_events GROUP BY device_id LIMIT 3;
EOF

echo ""
echo "Aggregated data (hourly statistics):"
cat <<'EOF' | docker exec -i cassandra-1 cqlsh
USE iot_analytics;
SELECT device_id, hour_bucket, avg_temperature, max_temperature, min_temperature, event_count 
FROM hourly_aggregates LIMIT 5;
EOF

=== RAW DATA IN CASSANDRA ===

Raw sensor events (recent first):

 device_id                            | timestamp     | temperature | humidity | location
--------------------------------------+---------------+-------------+----------+----------
 02a0fdf8-510e-43cd-b372-9ed343539cb0 | 1774572112603 |       19.59 |    79.72 |   Venice
 02a0fdf8-510e-43cd-b372-9ed343539cb0 | 1774572111604 |       34.55 |     54.8 |   Venice
 02a0fdf8-510e-43cd-b372-9ed343539cb0 | 1774572110604 |        19.2 |    71.66 |   Venice
 02a0fdf8-510e-43cd-b372-9ed343539cb0 | 1774572109604 |       18.83 |    67.85 |   Venice
 02a0fdf8-510e-43cd-b372-9ed343539cb0 | 1774572108603 |        28.7 |    83.74 |   Venice
 02a0fdf8-510e-43cd-b372-9ed343539cb0 | 1774572107603 |        21.5 |    54.42 |   Venice
 02a0fdf8-510e-43cd-b372-9ed343539cb0 | 1774572106603 |        22.7 |    71.86 |   Venice
 02a0fdf8-510e-43cd-b372-9ed343539cb0 | 1774572105605 |       29.91 |    61.19 |   Venice
 02a0fdf8-510e-43cd-b372-9ed34353

## Part 5: Key Takeaways

### Why Cassandra for Big Data?

1. **Distributed**: No single point of failure; add nodes for capacity
2. **Fast writes**: Append-only architecture; 100K+ events/sec per node
3. **Tunable consistency**: Trade consistency for latency when needed
4. **Horizontal scaling**: Linear performance improvement per node
5. **High availability**: Replication + gossip protocol

### The Trade-offs

**PROs**:
- Scale-Out -> Massive scale (petabytes)
- Tunabe Consistency -> High throughput (100K+ qps per node)
- Scale-out, P2P, Hash Ring -> No downtime (no data sharding, easy node additions and removal)
- NoSQL and Column-Family -> Flexible schema (columns are dynamic)

**COSTs**:
- No multi-row ACID transactions
- No joins (must denormalize data)
- Eventual consistency (temporary stale data)
- Operational complexity (monitoring, repairs)

### When to Use Cassandra

**Perfect for**: Time-series data, IoT sensor logs, analytics, user activity feeds  
**Avoid**: Transactional systems (banking), strong consistency required, small datasets